In [0]:
# 1: Setup and Imports
import gc
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, FeatureHasher
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

# 2: Load Data
df = spark.table("workspace.capstone_project.toronto_model_ready")

# ---------------------------------------------------------
# FULL NULL CLEANING
# ---------------------------------------------------------
categorical_cols = ['incident_category', 'season', 'unified_call_source', 'location_area']
numeric_cols = [
    'hour', 'day_of_week', 'month', 'year',
    'unified_alarm_level', 'calls_past_30min', 'calls_past_60min'
]

all_required_cols = ["delay_indicator"] + categorical_cols + numeric_cols

df = df.select(*all_required_cols)
df = df.dropna()

print("Row count after null filtering:", df.count())
df.groupBy("delay_indicator").count().show()

# ---------------------------------------------------------
# Class Weights
# ---------------------------------------------------------
class_counts = df.groupBy("delay_indicator").count().collect()
total = sum(r["count"] for r in class_counts)
weights = {r["delay_indicator"]: total / r["count"] for r in class_counts}

df = df.withColumn(
    "class_weight",
    F.when(F.col("delay_indicator") == 0, weights[0])
     .otherwise(weights[1])
)

# ---------------------------------------------------------
# Feature Hashing
# ---------------------------------------------------------
hasher = FeatureHasher(
    inputCols=categorical_cols,
    outputCol="categorical_features",
    numFeatures=1024
)

assembler = VectorAssembler(
    inputCols=numeric_cols + ["categorical_features"],
    outputCol="features"
)

# ---------------------------------------------------------
# Train/Test Split
# ---------------------------------------------------------
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# ---------------------------------------------------------
# Evaluators
# ---------------------------------------------------------
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="delay_indicator",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="delay_indicator",
    metricName="areaUnderPR"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="delay_indicator",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="delay_indicator",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="delay_indicator",
    predictionCol="prediction",
    metricName="f1"
)

# ---------------------------------------------------------
# Models (Balanced + Efficient)
# ---------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(
        featuresCol="features",
        labelCol="delay_indicator",
        weightCol="class_weight",
        maxIter=60
    ),

    "Random Forest": RandomForestClassifier(
        featuresCol="features",
        labelCol="delay_indicator",
        weightCol="class_weight",
        numTrees=150,
        maxDepth=10,
        minInstancesPerNode=10,
        subsamplingRate=0.8,
        seed=42
    ),

    "GBT Classifier": GBTClassifier(
        featuresCol="features",
        labelCol="delay_indicator",
        maxIter=60,
        maxDepth=6,
        stepSize=0.05,
        minInstancesPerNode=10,
        subsamplingRate=0.8,
        seed=42
    )
}

# ---------------------------------------------------------
# Train + Evaluate
# ---------------------------------------------------------
results = []

for model_name, classifier in models.items():
    pipeline = Pipeline(stages=[hasher, assembler, classifier])
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    auc = roc_evaluator.evaluate(predictions)
    auc_pr = pr_evaluator.evaluate(predictions)
    precision = precision_eval.evaluate(predictions)
    recall = recall_eval.evaluate(predictions)
    f1 = f1_eval.evaluate(predictions)

    results.append((model_name, auc, auc_pr, precision, recall, f1))

    del model
    del predictions
    del pipeline
    gc.collect()

# ---------------------------------------------------------
# Final Output Only
# ---------------------------------------------------------
print("\nFINAL MODEL PERFORMANCE SUMMARY - TORONTO")
print("=" * 90)
print(f"{'Model':<22} {'AUC-ROC':<12} {'PR-AUC':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12}")
print("-" * 90)

for row in results:
    model_name, auc, auc_pr, precision, recall, f1 = row
    print(f"{model_name:<22} {auc:<12.6f} {auc_pr:<12.6f} {precision:<12.6f} {recall:<12.6f} {f1:<12.6f}")